# MIC90 Prediction Using Multiple Machine Learning Models

## Dataset
input/MIC90_normalised_data.xlsx

## Preprocessing Sheets
1. Min-Max Scaling
2. MaxAbs Scaling
3. Log Transformation
4. Box-Cox Transformation

## Models

1. Multiple Linear Regression
2. LASSO Regression
3. Decision Tree Regressor
4. Random Forest Regressor
5. Support Vector Regressor
6. Gradient Boosting Regressor

## Evaluation Metrics

- R²
- Adjusted R²
- RMSE
- MAE
- CV_R²

In [1]:
# ============================================================
# IMPORT REQUIRED LIBRARIES
# ============================================================

import os

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import (
    LinearRegression,
    Lasso
)

from sklearn.tree import DecisionTreeRegressor

from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)

from sklearn.svm import SVR

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from openpyxl.styles import (
    Font,
    PatternFill,
    Border,
    Side,
    Alignment
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================================
# LOAD DATASET
# ============================================================

file_path = "input/MIC90_normalised_data.xlsx"

xls = pd.ExcelFile(file_path)

print("Worksheets Found:")

for sheet in xls.sheet_names:
    print(sheet)

Worksheets Found:
Min-Max Scaling
MaxAbs Scaling 
Log Transformation
Box-Cox Transformation


In [3]:
# ============================================================
# DEFINE MODELS
# ============================================================

models = {

    "Multiple linear regression":
        LinearRegression(),

    "LASSO regression":
        Lasso(
            alpha=0.01
        ),

    "Decision tree":
        DecisionTreeRegressor(
            max_depth=5,
            random_state=42
        ),

    "Random forest":
        RandomForestRegressor(
            n_estimators=300,
            random_state=42
        ),

    "Support vector machines":
        SVR(
            kernel="rbf",
            C=10
        ),

    "Gradient boosting":
        GradientBoostingRegressor(
            n_estimators=200,
            random_state=42
        )
}

print("Total Models =", len(models))

Total Models = 6


In [4]:
# ============================================================
# RUN ALL SHEETS
# ============================================================

all_results = []
all_predictions = []

for sheet in xls.sheet_names:

    print("\n")
    print("="*80)
    print("PROCESSING :", sheet)
    print("="*80)

    df = pd.read_excel(
        file_path,
        sheet_name=sheet
    )

    df.columns = [
        "MIC90",
        "LogP",
        "MR",
        "HL",
        "tPSA",
        "CLogP",
        "CMR",
        "LogS"
    ]

    X = df[
        [
            "LogP",
            "MR",
            "HL",
            "tPSA",
            "CLogP",
            "CMR",
            "LogS"
        ]
    ]

    y = df["MIC90"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(
        X_train
    )

    X_test_scaled = scaler.transform(
        X_test
    )

    prediction_df = pd.DataFrame()

    prediction_df["Actual_MIC90"] = y_test.values

    for name, model in models.items():

        if name in [
            "Multiple linear regression",
            "LASSO regression",
            "Support vector machines"
        ]:

            model.fit(
                X_train_scaled,
                y_train
            )

            y_pred = model.predict(
                X_test_scaled
            )

            cv_r2 = cross_val_score(
                model,
                X_train_scaled,
                y_train,
                cv=5,
                scoring="r2"
            ).mean()

        else:

            model.fit(
                X_train,
                y_train
            )

            y_pred = model.predict(
                X_test
            )

            cv_r2 = cross_val_score(
                model,
                X_train,
                y_train,
                cv=5,
                scoring="r2"
            ).mean()

        r2 = r2_score(
            y_test,
            y_pred
        )

        rmse = np.sqrt(
            mean_squared_error(
                y_test,
                y_pred
            )
        )

        mae = mean_absolute_error(
            y_test,
            y_pred
        )

        n = len(y_test)

        p = X.shape[1]

        adj_r2 = 1 - (
            ((1-r2)*(n-1)) /
            (n-p-1)
        )

        all_results.append([
            sheet,
            name,
            r2,
            adj_r2,
            rmse,
            mae,
            cv_r2
        ])

        prediction_df[name] = y_pred

    prediction_df["Sheet"] = sheet

    all_predictions.append(
        prediction_df
    )

print("All Sheets Processed")



PROCESSING : Min-Max Scaling


PROCESSING : MaxAbs Scaling 


PROCESSING : Log Transformation


PROCESSING : Box-Cox Transformation
All Sheets Processed


In [5]:
# ============================================================
# CREATE RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    all_results,
    columns=[
        "Sheet",
        "Model",
        "R2",
        "Adjusted_R2",
        "RMSE",
        "MAE",
        "CV_R2"
    ]
)

In [6]:
# ============================================================
# CREATE RANKINGS INSIDE EACH SHEET
# ============================================================

results_df["Rank"] = results_df.groupby(
    "Sheet"
)["R2"].rank(
    ascending=False,
    method="dense"
)

results_df = results_df.sort_values(
    [
        "Sheet",
        "Rank"
    ]
)

results_df

,Sheet,Model,R2,Adjusted_R2,RMSE,MAE,CV_R2,Rank
22,Box-Cox Transformation,Support vector machines,0.682859,0.606308,0.128550,0.083064,-0.028445,1.0
20,Box-Cox Transformation,Decision tree,0.627905,0.538088,0.139243,0.069390,0.145615,2.0
23,Box-Cox Transformation,Gradient boosting,0.625482,0.535082,0.139695,0.074722,0.205091,3.0
21,Box-Cox Transformation,Random forest,0.623149,0.532185,0.140130,0.070530,0.374326,4.0
18,Box-Cox Transformation,Multiple linear regression,0.494615,0.372625,0.162277,0.134161,0.111148,5.0
19,Box-Cox Transformation,LASSO regression,0.473702,0.346665,0.165600,0.137695,0.200594,6.0
17,Log Transformation,Gradient boosting,0.836057,0.796485,0.505016,0.246802,-2.699121,1.0
15,Log Transformation,Random forest,0.796438,0.747302,0.562740,0.248465,-2.478198,2.0
14,Log Transformation,Decision tree,0.790151,0.739498,0.571364,0.232344,-5.557890,3.0
16,Log Transformation,Support vector machines,0.786565,0.735046,0.576226,0.300052,-3.057244,4.0


In [7]:
# ============================================================
# BEST MODEL OF EACH PREPROCESSING METHOD
# ============================================================

results_df = results_df.sort_values(
    ["Sheet", "R2"],
    ascending=[True, False]
)

best_models = (
    results_df
    .groupby("Sheet")
    .first()
)

best_models

,Model,R2,Adjusted_R2,RMSE,MAE,CV_R2,Rank
Sheet,,,,,,,
Box-Cox Transformation,Support vector machines,0.682859,0.606308,0.128550,0.083064,-0.028445,1.0
Log Transformation,Gradient boosting,0.836057,0.796485,0.505016,0.246802,-2.699121,1.0
MaxAbs Scaling,Random forest,0.911058,0.889589,0.065757,0.031054,-331.307868,1.0
Min-Max Scaling,Random forest,0.909476,0.887626,0.066339,0.030891,-317.802759,1.0


In [8]:
# ============================================================
# BEST OVERALL MODEL
# ============================================================

best_overall = results_df.loc[
    results_df["R2"].idxmax()
]

print("BEST OVERALL MODEL")

best_overall

BEST OVERALL MODEL


Sheet          MaxAbs Scaling 
Model            Random forest
R2                    0.911058
Adjusted_R2           0.889589
RMSE                  0.065757
MAE                   0.031054
CV_R2              -331.307868
Rank                       1.0
Name: 9, dtype: object

In [9]:
# ============================================================
# COMBINE ALL PREDICTIONS
# ============================================================

predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)

predictions_df.head()

,Actual_MIC90,Multiple linear regression,LASSO regression,Decision tree,Random forest,Support vector machines,Gradient boosting,Sheet
0,0.107959,0.088975,0.092684,0.000263,0.000501,0.130559,0.002769,Min-Max Scaling
1,0.008571,0.045436,0.011597,0.006632,0.004966,0.132754,0.005300,Min-Max Scaling
2,0.000328,0.023591,0.059884,0.000263,0.007465,-0.083590,0.002857,Min-Max Scaling
3,0.000034,0.042313,0.051550,0.000263,0.000438,-0.016123,0.000084,Min-Max Scaling
4,0.000333,0.066312,0.090457,0.000263,0.017118,-0.109865,0.006120,Min-Max Scaling


In [10]:
# ============================================================
# EXPORT TO EXCEL
# ============================================================

os.makedirs("output/workbook", exist_ok=True)

with pd.ExcelWriter(
    "output/workbook/MIC90_Results.xlsx",
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Results",
        index=False
    )

    workbook = writer.book
    worksheet = writer.sheets["Results"]

    for cell in worksheet[1]:
        cell.font = Font(
            bold=True
        )
        
        cell.border = Border(
            left=Side(style="thin"),
            right=Side(style="thin"),
            top=Side(style="thin"),
            bottom=Side(style="thin")
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center"
        )
        
    for row in worksheet.iter_rows():

        for cell in row:

            cell.border = Border(
                left=Side(style="thin"),
                right=Side(style="thin"),
                top=Side(style="thin"),
                bottom=Side(style="thin")
            )

    headers = [
        cell.value
        for cell in worksheet[1]
    ]

    if "Rank" in headers:

        rank_col = headers.index("Rank") + 1

        for row in range(2, worksheet.max_row + 1):

            rank_value = worksheet.cell(
                row=row,
                column=rank_col
            ).value

            if rank_value == 1:

                for col in range(1,worksheet.max_column + 1):

                    worksheet.cell(
                        row=row,
                        column=col
                    ).fill = PatternFill(
                        start_color = "C6EFCE",
                        end_color = "C6EFCE",
                        fill_type="solid"
                    )

In [11]:
# ============================================================
# EXPORT PREDICTIONS CSV
# ============================================================

predictions_df.to_csv(
    "output/workbook/MIC90_Predictions.csv",
    index=False
)

print("Predictions CSV Saved")

Predictions CSV Saved


In [12]:
# ============================================================
# SHOW FINAL RANKED TABLE
# ============================================================

results_df

,Sheet,Model,R2,Adjusted_R2,RMSE,MAE,CV_R2,Rank
22,Box-Cox Transformation,Support vector machines,0.682859,0.606308,0.128550,0.083064,-0.028445,1.0
20,Box-Cox Transformation,Decision tree,0.627905,0.538088,0.139243,0.069390,0.145615,2.0
23,Box-Cox Transformation,Gradient boosting,0.625482,0.535082,0.139695,0.074722,0.205091,3.0
21,Box-Cox Transformation,Random forest,0.623149,0.532185,0.140130,0.070530,0.374326,4.0
18,Box-Cox Transformation,Multiple linear regression,0.494615,0.372625,0.162277,0.134161,0.111148,5.0
19,Box-Cox Transformation,LASSO regression,0.473702,0.346665,0.165600,0.137695,0.200594,6.0
17,Log Transformation,Gradient boosting,0.836057,0.796485,0.505016,0.246802,-2.699121,1.0
15,Log Transformation,Random forest,0.796438,0.747302,0.562740,0.248465,-2.478198,2.0
14,Log Transformation,Decision tree,0.790151,0.739498,0.571364,0.232344,-5.557890,3.0
16,Log Transformation,Support vector machines,0.786565,0.735046,0.576226,0.300052,-3.057244,4.0
